# UniGoal 室內導航 — Colab 版
**執行順序**：依序跑 Setup 區塊（只需跑一次），再跑導航區塊。

> ⚠️ 每次重連 Colab Runtime 都要重跑 Setup（Ollama 和模型不會保留）。
> 照片放在 Google Drive 只需上傳一次。

## 🔧 Setup — 只需跑一次

In [ ]:
# 1. Clone repo
import os
if not os.path.exists('/content/NAVIGATION-GroundingDINO-'):
    !git clone https://github.com/B1222017/NAVIGATION-GroundingDINO-.git
%cd /content/NAVIGATION-GroundingDINO-
print('✅ Repo ready')

In [ ]:
# 2. 安裝 Python 套件
!pip install -q \
    'transformers>=4.50.0' \
    'torch>=2.2.0' \
    'torchvision' \
    'easyocr' \
    'Pillow' \
    'matplotlib' \
    'networkx' \
    'requests'
print('✅ Packages installed')

In [ ]:
# 3. 安裝 Ollama
!curl -fsSL https://ollama.ai/install.sh | sh
print('✅ Ollama installed')

In [ ]:
# 4. 啟動 Ollama service（背景執行）
import subprocess, time
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)
# 確認是否啟動
import requests
try:
    r = requests.get('http://127.0.0.1:11434', timeout=5)
    print('✅ Ollama service running')
except:
    print('❌ Ollama 未啟動，請重跑這格')

In [ ]:
# 5. 下載 VLM 模型（第一次較久，約 5-10 分鐘）
# gemma4 若找不到可改成 gemma3:4b 或 llama3.2-vision:11b
MODEL = 'gemma4:latest'
!ollama pull {MODEL}
print(f'✅ Model {MODEL} ready')

In [ ]:
# 6. 掛載 Google Drive（照片存放處）
from google.colab import drive
drive.mount('/content/drive')

# ⚙️ 修改成你在 Google Drive 裡的照片資料夾路徑
PHOTO_DIR = '/content/drive/MyDrive/系辦照片'

import os
if os.path.exists(PHOTO_DIR):
    photos = [f for f in os.listdir(PHOTO_DIR) if f.endswith('.jpg')]
    print(f'✅ 找到 {len(photos)} 張照片')
else:
    print(f'❌ 路徑不存在：{PHOTO_DIR}')
    print('請先把照片上傳到 Google Drive，再修改上方 PHOTO_DIR')

## 🧭 導航執行

In [ ]:
# 重設 session（新的導航任務前執行）
!python3 navigate_one_by_one.py --reset

In [ ]:
# ⚙️ 設定目標
GOAL = '電腦'  # 改成你要找的東西

# 照片清單與對應輸入文字（按走路順序）
STEPS = [
    ('IMG_1433', '剛進大廳，我該往哪個方向走才能找到電腦？'),
    ('IMG_1435', '沿著接待區旁邊走，我應該往走廊深處去嗎？'),
    ('IMG_1438', '走廊兩側都是辦公室，我應該繼續直走嗎？'),
    ('IMG_1441', '前方還是走廊，有沒有走錯方向？'),
    ('IMG_1498', '看到很多教師辦公室，電腦可能在這附近嗎？'),
    ('IMG_1500', '繼續往前，這條路有什麼新的標示？'),
    ('IMG_1502', '看到會議室了，要繼續直走還是轉彎？'),
    ('IMG_1503', '走廊還在延伸，左右有沒有出口？'),
    ('IMG_1506', '這裡有其他岔路嗎，還是繼續直走？'),
    ('IMG_1510', '這個位置有書架跟辦公桌，有看到電腦在哪嗎？'),
    ('IMG_1511', '繼續往前走，這個方向感覺對嗎？'),
    ('IMG_1513', '看到一排看起來像儲物區的地方，電腦會在這嗎？'),
    ('IMG_1471', '走廊右邊有木製儲物櫃，我該怎麼繼續走？'),
    ('IMG_1468', '一整排儲物櫃，電腦不太可能在這裡，要折返嗎？'),
    ('IMG_1467', '這裡只有辦公室的門，要繼續走嗎？'),
    ('IMG_1516', '看到會客室的標示，方向是這邊嗎？'),
    ('IMG_1517', '這裡有教授辦公室門牌，電腦區通常在哪個方向？'),
    ('IMG_1518', '繼續沿走廊走，有沒有看到電腦設備？'),
    ('IMG_1521', '走廊感覺好長，有沒有任何指向電腦的線索？'),
    ('IMG_1522', '牆上有海報，有沒有看到任何指示牌？'),
    ('IMG_1523', '好像回到大廳區域了，有電腦設備嗎？'),
    ('IMG_1524', '這邊有接待桌，電腦設備在服務台附近嗎？'),
    ('IMG_1526', '還在走廊，我是不是走錯路了？'),
    ('IMG_1528', '前面看起來有設備區，這裡是電腦所在嗎？'),
]

print(f'目標：{GOAL}，共 {len(STEPS)} 步')
print('請逐格執行下方的「執行一步」格，每步完成後再執行下一步。')

In [ ]:
# 執行一步（每次跑這格就前進一張照片）
# 偵測到目標時會出現 y/s/n/Enter 的確認提示，直接在下方輸入即可

import json
with open('session_navigate.json') as f:
    _s = json.load(f)
current_step = _s.get('step', 0)

if current_step >= len(STEPS):
    print('✅ 所有步驟已完成')
else:
    img, text = STEPS[current_step]
    photo_path = f'{PHOTO_DIR}/{img}.jpg'
    print(f'▶ Step {current_step + 1}/{len(STEPS)}: {img}')
    %run navigate_one_by_one.py --goal {GOAL} --photo "{photo_path}" --text "{text}"

In [ ]:
# （選用）手動標記上一步的結果
# 若跳過確認後想補標，執行此格
MARK = 's'  # 'y'=找到, 's'=同種非目標, 'n'=誤判

import json
with open('session_navigate.json') as f:
    s = json.load(f)
last_step = s['step']
for obs in s['observations']:
    if obs['step'] == last_step:
        if MARK == 'y':
            obs['arrived_here'] = True
            s['arrived'] = True
        elif MARK == 's':
            obs['wrong_instance'] = True
            s['corrections'].append(f'步驟 {last_step}：同種非目標，繼續搜索')
        elif MARK == 'n':
            obs['false_positive'] = True
            s['false_positives'].append(last_step)
            s['corrections'].append(f'步驟 {last_step}：誤判，繼續搜索')
with open('session_navigate.json', 'w') as f:
    json.dump(s, f, indent=2, ensure_ascii=False)
print(f'✅ 步驟 {last_step} 標記為 {MARK}')

In [ ]:
# 重新渲染最終場景圖
import json, sys
sys.path.insert(0, '.')
from navigate_one_by_one import setup_goal, render_scene_graph
with open('session_navigate.json') as f:
    s = json.load(f)
setup_goal(GOAL)
sg = render_scene_graph(s['observations'], s['step'], s.get('corrections', []))

from IPython.display import Image
display(Image(str(sg)))